# 03 · Final Model Selection & Training (S6)

**Course:** Data Mining (DA) — Autumn 2026 · **Group 7**  
**Topic:** Credit Card Fraud Detection (Option E)

## Mục tiêu S6

S5 đã benchmark **27 tổ hợp** và tìm ra **tie** ở operating point P≥95%:

| Cấu hình | Recall@P95 | Cost | AUC-PR | F1@0.5 | #flag@0.5 |
|---|---|---|---|---|---|
| RF + SMOTE (1:100) | **0.7817** | $170.45 | **0.8239** | **0.8605** | 116 |
| RF + no resampling | **0.7817** | $170.45 | 0.8055 | 0.8425 | 112 |

S6 làm 4 việc:

1. **Chốt model chính thức** theo tiêu chí S6.4 (recall@P95 → AUC-PR → F1 → độ đơn giản/tốc độ)
2. **Refine hyperparameter** sâu hơn cho 2 model top (S5 dùng grid nhỏ để tiết kiệm thời gian)
3. **Train lại model cuối** trên toàn bộ train → lưu `final_model.joblib` kèm metadata
4. **So sánh sâu 3 model top**: PR/ROC overlay, calibration, latency

> ⏱️ **Ước tính: 30–50 phút** (XGBoost nhanh, Random Forest chậm nhất).

In [1]:
import sys, warnings, time, json, platform
warnings.filterwarnings("ignore")

from pathlib import Path
PROJECT_ROOT = Path(r"e:\Projects\DataMining")
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np, pandas as pd, joblib
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (SEED, PALETTE, COLORS, TABLES, CHARTS, MODELS, MIN_PRECISION,
                        COST_PER_FN, COST_PER_FP, DATASET_DAYS)
from src.modeling import (make_pipeline, MODEL_LABELS, IMBALANCE_LABELS,
                          fit_isolation_forest)
from src.evaluation import (evaluate, confusion_counts, find_threshold,
                            accuracy_paradox, measure_latency, threshold_curve,
                            plot_pr_overlay, plot_roc_overlay, plot_reliability,
                            plot_confusion_matrix)

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import brier_score_loss

sns.set_style("whitegrid")
plt.rcParams.update({"figure.figsize": (14, 6), "font.family": "DejaVu Sans"})

FINAL_CSV = TABLES / "final_model_comparison.csv"
print("env   :", platform.python_version(), "| sklearn", __import__("sklearn").__version__)
print("seed  :", SEED)
print("table :", FINAL_CSV)

env   : 3.11.9 | sklearn 1.9.1
seed  : 42
table : E:\Projects\DataMining\artifacts\tables\final_model_comparison.csv


In [2]:
d = joblib.load(MODELS / "split_with_features.joblib")
X_train, X_test = d["X_train"], d["X_test"]
y_train, y_test = d["y_train"], d["y_test"]
COLS = list(X_train.columns)

print("train:", X_train.shape, "| test:", X_test.shape)
print("fraud train:", int(y_train.sum()), "| fraud test:", int(y_test.sum()))

# Load ket qua S5
bench = pd.read_csv(TABLES / "imbalance_benchmark.csv").drop_duplicates("key", keep="last")
best_cfg = joblib.load(MODELS / "s5_best_config.joblib")

def _lbl(cfg_part):
    """Label an toan: uu tien imbalance_label, fallback sang imbalance."""
    return cfg_part.get("imbalance_label") or IMBALANCE_LABELS.get(cfg_part.get("imbalance"), "?")

print("\nS5 winner:", best_cfg["primary"]["model_label"], "+",
      _lbl(best_cfg["primary"]), "strat=", best_cfg["primary"]["strategy"])
print("S5 runner-up:", best_cfg["runner_up"]["model_label"], "+",
      _lbl(best_cfg["runner_up"]))
print("tie group size:", len(best_cfg["tied_at_operating_point"]))
for t in best_cfg["tied_at_operating_point"]:
    print(f"   - {t.get('model_label', '?')} + {t.get('imbalance_label', '?')} "
          f"| strat={t.get('strategy')} | recall={t.get('recall', float('nan')):.4f}")

train: (198608, 35) | test: (85118, 35)
fraud train: 331 | fraud test: 142

S5 winner: Random Forest + SMOTE strat= 0.01
S5 runner-up: Random Forest + No resampling
tie group size: 2
   - Random Forest + SMOTE | strat=0.01 | recall=0.7817
   - Random Forest + No resampling | strat=None | recall=0.7817


## 1. Tiêu chí chọn model (S6.4 — ghi rõ trong report)

| Ưu tiên | Tiêu chí | Lý do |
|---|---|---|
| **1** | **Recall @ Precision ≥ 0.95** | Đúng câu hỏi kinh doanh #2 của Nhóm 1 |
| **2** | **AUC-PR** | Nhạy với imbalance hơn AUC-ROC |
| **3** | F1-score | Cân bằng hai loại lỗi |
| **4** | Tính đơn giản + tốc độ suy luận | Không phải yếu tố quyết định, nhưng cần đo để trả lời "nhân rộng được không?" |

> ❌ **Không chọn theo accuracy** — accuracy 99.83% là của model "luôn nói legit".

In [3]:
# Bang tong hop S5 -> dua 3 model top vao so sanh sau
cand = bench.dropna(subset=["recall_p95"]).copy()
cand = cand.sort_values(
    ["recall_p95", "f1_p95", "auc_pr"], ascending=[False, False, False]
).reset_index(drop=True)

print("=== TOP 6 theo recall@P95 (test set) ===")
print(
    cand.head(6)[["model_label", "imbalance_label", "strategy", "thr_p95",
               "precision_p95", "recall_p95", "f1_p95", "auc_pr", "auc_roc"]]
    .round(4).to_string(index=False)
)

# 3 model se dua vao deep comparison
TOP3 = [
    ("rf", "smote", 0.01),      # winner S5
    ("rf", "none", None),        # runner-up S5
    ("xgb", "smote", 0.05),      # nhanh nhat, AUC-PR tuong duong
]
print("\nDeep comparison set:", TOP3)

=== TOP 6 theo recall@P95 (test set) ===
  model_label imbalance_label  strategy  thr_p95  precision_p95  recall_p95  f1_p95  auc_pr  auc_roc
Random Forest           SMOTE      0.01   0.5200         0.9569      0.7817  0.8605  0.8239   0.9567
Random Forest   No resampling       NaN   0.4833         0.9569      0.7817  0.8605  0.8055   0.9501
Random Forest           SMOTE      0.05   0.5767         0.9558      0.7606  0.8471  0.8336   0.9806
      XGBoost           SMOTE      0.10   0.7015         0.9558      0.7606  0.8471  0.8111   0.9788
Random Forest           SMOTE      0.10   0.6067         0.9554      0.7535  0.8425  0.8348   0.9759
      XGBoost           SMOTE      0.05   0.7222         0.9554      0.7535  0.8425  0.8244   0.9761

Deep comparison set: [('rf', 'smote', 0.01), ('rf', 'none', None), ('xgb', 'smote', 0.05)]


## 2. Refine hyperparameter cho 2 model top

S5 dùng grid **nhỏ** (để cả 27 tổ hợp xong trong 73 phút). S6 mở rộng grid
cho 2 model top — đây là bước **tinh chỉnh cuối cùng**.

| Model | Grid S5 | Grid S6 (mở rộng) |
|---|---|---|
| Random Forest | `max_depth` [10, None] × `min_samples_leaf` [1, 10] | `max_depth` [None, 12, 20] × `min_samples_leaf` [1, 2, 5] × `n_estimators` [300, 600] |
| XGBoost | `max_depth` [4, 6] × `n_estimators` [200, 400] | `max_depth` [4, 6, 8] × `learning_rate` [0.05, 0.1] × `n_estimators` [300, 600] × `min_child_weight` [1, 5] |

> Vẫn dùng `scoring="average_precision"` và CV **trên train**.

In [4]:
GRIDS_S6 = {
    "rf": {
        "clf__max_depth": [None, 12, 20],
        "clf__min_samples_leaf": [1, 2, 5],
        "clf__n_estimators": [300, 600],
    },
    "xgb": {
        "clf__max_depth": [4, 6, 8],
        "clf__learning_rate": [0.05, 0.1],
        "clf__n_estimators": [300, 600],
        "clf__min_child_weight": [1, 5],
    },
}

for k, v in GRIDS_S6.items():
    n = 1
    for vals in v.values():
        n *= len(vals)
    print(f"{k}: {n} combos x 3 folds")

rf: 18 combos x 3 folds
xgb: 24 combos x 3 folds


In [5]:
def refine(model, imbalance, strategy, grid=None, n_splits=3):
    """GridSearchCV tren train -> refit -> danh gia 1 lan tren test."""
    t0 = time.time()
    pipe = make_pipeline(
        model, imbalance=imbalance, sampler_strategy=strategy,
        scaler="standard", feature_cols=COLS,
    )
    if grid:
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
        gs = GridSearchCV(pipe, grid, cv=skf, scoring="average_precision",
                          n_jobs=1, refit=True, verbose=0)
        gs.fit(X_train, y_train)
        pipe = gs.best_estimator_
        best_params = gs.best_params_
        cv_ap = float(gs.best_score_)
    else:
        pipe.fit(X_train, y_train)
        best_params = {}
        cv_ap = np.nan

    y_score = pipe.predict_proba(X_test)[:, 1]
    y_pred = (y_score >= 0.5).astype(int)
    met = evaluate(y_test, y_pred, y_score)
    thr = find_threshold(y_test, y_score, MIN_PRECISION)
    cc = confusion_counts(y_test, y_pred)

    return {
        "model": model, "imbalance": imbalance, "strategy": strategy,
        "model_label": MODEL_LABELS[model],
        "imbalance_label": IMBALANCE_LABELS[imbalance],
        "cv_ap": cv_ap,
        "best_params": json.dumps(best_params, default=str),
        **met,
        "tp": cc["tp"], "fp": cc["fp"], "fn": cc["fn"], "tn": cc["tn"],
        "thr_p95": thr["threshold"] if thr else np.nan,
        "precision_p95": thr["precision"] if thr else np.nan,
        "recall_p95": thr["recall"] if thr else np.nan,
        "f1_p95": thr["f1"] if thr else np.nan,
        "cost_at_0.5": cc["fn"] * COST_PER_FN + cc["fp"] * COST_PER_FP,
        "brier": float(brier_score_loss(y_test, y_score)),
        "seconds": round(time.time() - t0, 1),
    }, pipe, y_score

## 3. ⭐ Chạy refinement (RF + SMOTE 1:100, XGBoost + SMOTE 1:20)

**Cell này chạy lâu (~30–45 phút)** vì GridSearchCV cho Random Forest có tới
18 tổ hợp × 3 folds, mỗi fold phải SMOTE trên 132K dòng.

> ⚠️ Kết quả được **append vào CSV ngay lập tức** → nếu kernel chết, chạy lại
> sẽ bỏ qua phần đã xong.

In [6]:
REFINE_TASKS = [
    ("rf", "smote", 0.01, GRIDS_S6["rf"]),
    ("xgb", "smote", 0.05, GRIDS_S6["xgb"]),
]

if FINAL_CSV.exists():
    prev = pd.read_csv(FINAL_CSV)
    print(f"Resume: {len(prev)} configs already refined")
else:
    prev = pd.DataFrame()
    print("Fresh run")

done_keys = set(prev["key"]) if "key" in prev.columns else set()

for i, (m, imb, st, grid) in enumerate(REFINE_TASKS, 1):
    key = f"{m}__{imb}__{st if st is not None else 'na'}"
    if key in done_keys:
        print(f"[{i}/{len(REFINE_TASKS)}] {key} — skip (done)")
        continue
    print(f"[{i}/{len(REFINE_TASKS)}] {key} ...", flush=True)
    row, pipe, y_score = refine(m, imb, st, grid)
    row["key"] = key
    row["stage"] = "S6_refined"
    joblib.dump(pipe, MODELS / f"s6_{key}.joblib")
    np.save(MODELS / "s5_predictions" / f"s6_{key}.npy", y_score)
    pd.DataFrame([row]).to_csv(FINAL_CSV, mode="a",
                            header=not FINAL_CSV.exists(), index=False)
    print(f"     CV-AP={row['cv_ap']:.4f} F1={row['f1']:.4f} "
          f"AP={row['auc_pr']:.4f} R@P95={row['recall_p95']:.4f} "
          f"({row['seconds']}s)")

print("\nDONE")

Resume: 2 configs already refined
[1/2] rf__smote__0.01 — skip (done)
[2/2] xgb__smote__0.05 — skip (done)

DONE


In [7]:
final = pd.read_csv(FINAL_CSV).drop_duplicates("key", keep="last")
final = final.sort_values(["recall_p95", "f1_p95", "auc_pr"],
                         ascending=[False, False, False]).reset_index(drop=True)
final.to_csv(FINAL_CSV, index=False)

print("=== KET QUA REFINEMENT S6 ===")
display(
    final[["model_label", "imbalance_label", "strategy", "cv_ap", "auc_pr",
           "auc_roc", "f1", "recall_p95", "precision_p95", "n_pred_pos",
           "brier", "seconds"]].round(4).reset_index(drop=True)
)

for _, r in final.iterrows():
    print(f"\n--- {r['model_label']} + {r['imbalance_label']} (strat={r['strategy']}) ---")
    print("  best_params:", r["best_params"])
    print(f"  CV-AP={r['cv_ap']:.4f} | test AUC-PR={r['auc_pr']:.4f} | "
          f"AUC-ROC={r['auc_roc']:.4f}")
    print(f"  @0.5  F1={r['f1']:.4f} P={r['precision']:.4f} R={r['recall']:.4f} "
          f"flag={r['n_pred_pos']}")
    print(f"  @P95  thr={r['thr_p95']:.4f} P={r['precision_p95']:.4f} "
          f"R={r['recall_p95']:.4f} F1={r['f1_p95']:.4f}")

=== KET QUA REFINEMENT S6 ===


,model_label,imbalance_label,strategy,cv_ap,auc_pr,auc_roc,f1,recall_p95,precision_p95,n_pred_pos,brier,seconds
0,Random Forest,SMOTE,0.01,0.8547,0.8234,0.9742,0.8605,0.7817,0.9569,116,0.0004,2447.8
1,XGBoost,SMOTE,0.05,0.8689,0.8198,0.9765,0.8295,0.7465,0.9550,116,0.0005,148.3



--- Random Forest + SMOTE (strat=0.01) ---
  best_params: {"clf__max_depth": 20, "clf__min_samples_leaf": 1, "clf__n_estimators": 600}
  CV-AP=0.8547 | test AUC-PR=0.8234 | AUC-ROC=0.9742
  @0.5  F1=0.8605 P=0.9569 R=0.7817 flag=116
  @P95  thr=0.5233 P=0.9569 R=0.7817 F1=0.8605

--- XGBoost + SMOTE (strat=0.05) ---
  best_params: {"clf__learning_rate": 0.1, "clf__max_depth": 8, "clf__min_child_weight": 5, "clf__n_estimators": 600}
  CV-AP=0.8689 | test AUC-PR=0.8198 | AUC-ROC=0.9765
  @0.5  F1=0.8295 P=0.9224 R=0.7535 flag=116
  @P95  thr=0.8048 P=0.9550 R=0.7465 F1=0.8379


## 4. So sánh sâu 3 model top

### 4.1 Baseline: accuracy paradox (để nhấn mạnh vì sao không dùng accuracy)

In [8]:
ap = accuracy_paradox(y_test)

print("=== ACCURACY PARADOX — model 'luon noi hop le' ===")
for k, v in ap.items():
    print(f"  {k:12s}: {v:.4f}")

print("\n=> Model nao KHONG bat duoc mot giao dich gian lan nao, van dat")
print(f"   {ap['accuracy']:.2%} accuracy. Do la ly do bao cao phai dung")
print("   Recall / Precision / F1 / AUC-PR, KHONG dung accuracy.")

=== ACCURACY PARADOX — model 'luon noi hop le' ===
  accuracy    : 0.9983
  recall      : 0.0000
  precision   : 0.0000
  f1          : 0.0000

=> Model nao KHONG bat duoc mot giao dich gian lan nao, van dat
   99.83% accuracy. Do la ly do bao cao phai dung
   Recall / Precision / F1 / AUC-PR, KHONG dung accuracy.


In [9]:
# Gom y_score cua 3 model top (dung file .npy da luu)
PRED_DIR = MODELS / "s5_predictions"

score_map = {}
for key, label in [
    ("rf__smote__0.01", "RF + SMOTE (1:100)"),
    ("rf__none__na",    "RF + no resampling"),
    ("xgb__smote__0.05","XGBoost + SMOTE (1:20)"),
]:
    p = PRED_DIR / f"{key}.npy"
    if p.exists():
        score_map[label] = np.load(p)
    else:
        print("MISSING:", p)

# them ban refined neu co
for _, r in final.iterrows():
    lab = f"{r['model_label']} + {r['imbalance_label']} (refined)"
    p = PRED_DIR / f"s6_{r['key']}.npy"
    if p.exists():
        score_map[lab] = np.load(p)

print("Models in comparison:")
for k in score_map:
    print("  -", k)

Models in comparison:
  - RF + SMOTE (1:100)
  - RF + no resampling
  - XGBoost + SMOTE (1:20)
  - Random Forest + SMOTE (refined)
  - XGBoost + SMOTE (refined)


In [10]:
plot_pr_overlay(y_test, score_map, CHARTS / "s6_pr_overlay.png",
               title="Precision–Recall — 3 candidate models")
plot_roc_overlay(y_test, score_map, CHARTS / "s6_roc_overlay.png",
                 title="ROC — 3 candidate models")
print("charts saved")

[eval] chart saved -> E:\Projects\DataMining\artifacts\charts\s6_pr_overlay.png


[eval] chart saved -> E:\Projects\DataMining\artifacts\charts\s6_roc_overlay.png
charts saved


### 4.2 Calibration — xác suất dự đoán có đáng tin không?

Điều này quyết định việc ngân hàng có thể gọi `predict_proba` là **"xác suất
rủi ro"** hay không. Nếu model over-confident (dưới đường chéo), threshold
0.52 **không** có nghĩa là "52% khả năng gian lận".

In [11]:
from sklearn.calibration import calibration_curve

print("=== CALIBRATION (quantile bins, 10 bins) ===")
cal_rows = []
for name, sc in score_map.items():
    pt, pp = calibration_curve(y_test, sc, n_bins=10, strategy="quantile")
    brier = brier_score_loss(y_test, sc)
    # sai lech trung binh giua xac suat du doan va ty le quan sat
    gap = float(np.mean(np.abs(pp - pt)))
    cal_rows.append({"model": name, "brier": brier, "mean_abs_gap": gap})
    print(f"  {name:40s} Brier={brier:.4f}  mean|gap|={gap:.4f}")

cal_df = pd.DataFrame(cal_rows)
print("\n(Brier score: cang thap cang tot)")

=== CALIBRATION (quantile bins, 10 bins) ===
  RF + SMOTE (1:100)                       Brier=0.0004  mean|gap|=0.0048
  RF + no resampling                       Brier=0.0005  mean|gap|=0.0007
  XGBoost + SMOTE (1:20)                   Brier=0.0005  mean|gap|=0.0002
  Random Forest + SMOTE (refined)          Brier=0.0004  mean|gap|=0.0008
  XGBoost + SMOTE (refined)                Brier=0.0005  mean|gap|=0.0002

(Brier score: cang thap cang tot)


In [12]:
best_name = list(score_map.keys())[0]
plot_reliability(y_test, score_map[best_name], CHARTS / "s6_reliability.png",
                 label=best_name)
print("saved")

[eval] chart saved -> E:\Projects\DataMining\artifacts\charts\s6_reliability.png
saved


### 4.3 Latency — có nhân rộng được không? (câu hỏi kinh doanh #6 của Nhóm 3)

Đo trên pipeline đã train, dùng 1.000 giao dịch, trung bình 5 lần.

In [13]:
print("=== LATENCY (1000 transactions x 5 repeats) ===")
lat_rows = []

for m, imb, st in [("rf", "smote", 0.01), ("rf", "none", None), ("xgb", "smote", 0.05)]:
    key = f"{m}__{imb}__{st if st is not None else 'na'}"
    p = MODELS / f"s5_{key}.joblib"
    if not p.exists():
        print("missing model:", p.name)
        continue
    pipe = joblib.load(p)
    lat = measure_latency(pipe, X_test, n_samples=1000, repeats=5)
    lat_rows.append({"model": f"{MODEL_LABELS[m]} + {IMBALANCE_LABELS[imb]}", **lat})
    print(f"  {MODEL_LABELS[m]:18s} + {IMBALANCE_LABELS[imb]:22s} "
          f"{lat['ms_per_transaction']:.3f} ms/txn  "
          f"({lat['cpu_hours_per_1M_txns']:.3f} CPU-h / 1M txns)")

lat_df = pd.DataFrame(lat_rows)
lat_df.to_csv(TABLES / "s6_latency.csv", index=False)
print("\nsaved ->", TABLES / "s6_latency.csv")

=== LATENCY (1000 transactions x 5 repeats) ===


  Random Forest      + SMOTE                  0.044 ms/txn  (0.012 CPU-h / 1M txns)


  Random Forest      + No resampling          0.039 ms/txn  (0.011 CPU-h / 1M txns)
  XGBoost            + SMOTE                  0.004 ms/txn  (0.001 CPU-h / 1M txns)

saved -> E:\Projects\DataMining\artifacts\tables\s6_latency.csv


## 5. ⭐ Chốt model chính thức & lưu `final_model.joblib`

Áp dụng đúng tiêu chí S6.4: recall@P95 → AUC-PR → F1 → độ đơn giản/tốc độ.

In [14]:
# Gom tat ca candidate: S5 refined + S5 baseline cua 3 model top
all_rows = []

for _, r in final.iterrows():
    rr = r.to_dict()
    rr["source"] = "S6_refined"
    all_rows.append(rr)

for m, imb, st in TOP3:
    key = f"{m}__{imb}__{st if st is not None else 'na'}"
    hit = bench[bench["key"] == key]
    if len(hit):
        rr = hit.iloc[0].to_dict()
        rr["source"] = "S5"
        all_rows.append(rr)

cand_all = pd.DataFrame(all_rows)
cand_all = cand_all.dropna(subset=["recall_p95"])
cand_all = cand_all.sort_values(
    ["recall_p95", "f1_p95", "auc_pr"], ascending=[False, False, False]
).reset_index(drop=True)

print("=== FINAL CANDIDATE RANKING ===")
print(
    cand_all[["model_label", "imbalance_label", "strategy", "source",
              "thr_p95", "precision_p95", "recall_p95", "f1_p95",
              "auc_pr", "auc_roc", "n_pred_pos"]].round(4).to_string(index=False)
)

winner = cand_all.iloc[0]
tie = cand_all[cand_all["recall_p95"] >= cand_all["recall_p95"].max() - 1e-9]

print("=" * 70)
print("WINNER:", winner["model_label"], "+", winner["imbalance_label"],
      "| strategy =", winner["strategy"], "| source =", winner["source"])
print(f"  recall@P95 = {winner['recall_p95']:.4f} @ threshold {winner['thr_p95']:.4f}")
print(f"  precision  = {winner['precision_p95']:.4f}")
print(f"  AUC-PR     = {winner['auc_pr']:.4f} | AUC-ROC = {winner['auc_roc']:.4f}")
print(f"\nTIE group: {len(tie)} configurations")
print(tie[["model_label", "imbalance_label", "strategy", "source",
           "recall_p95", "f1_p95", "auc_pr"]].round(4).to_string(index=False))

=== FINAL CANDIDATE RANKING ===
  model_label imbalance_label  strategy     source  thr_p95  precision_p95  recall_p95  f1_p95  auc_pr  auc_roc  n_pred_pos
Random Forest           SMOTE      0.01         S5   0.5200         0.9569      0.7817  0.8605  0.8239   0.9567         116
Random Forest           SMOTE      0.01 S6_refined   0.5233         0.9569      0.7817  0.8605  0.8234   0.9742         116
Random Forest   No resampling       NaN         S5   0.4833         0.9569      0.7817  0.8605  0.8055   0.9501         112
      XGBoost           SMOTE      0.05         S5   0.7222         0.9554      0.7535  0.8425  0.8244   0.9761         114
      XGBoost           SMOTE      0.05 S6_refined   0.8048         0.9550      0.7465  0.8379  0.8198   0.9765         116
WINNER: Random Forest + SMOTE | strategy = 0.01 | source = S5
  recall@P95 = 0.7817 @ threshold 0.5200
  precision  = 0.9569
  AUC-PR     = 0.8239 | AUC-ROC = 0.9567

TIE group: 3 configurations
  model_label imbalance_label

In [15]:
# Train lai model chinh thuc tren TOAN BO train, luu kem metadata
win_model = winner["model"]
win_imb = winner["imbalance"]
win_strat = None if pd.isna(winner["strategy"]) else float(winner["strategy"])

print(f"Training final model: {win_model} + {win_imb} (strat={win_strat})")

final_pipe = make_pipeline(
    win_model, imbalance=win_imb, sampler_strategy=win_strat,
    scaler="standard", feature_cols=COLS,
)

# Apply hyperparameter da chon tren TRAIN (tu GridSearchCV) — tuong minh
# chung, khong "lat" gi tri tu test set.
win_params = json.loads(winner.get("best_params") or "{}")
if win_params:
    final_pipe.set_params(**win_params)
    print("Applied best_params (selected on TRAIN via GridSearchCV):")
    for k, v in win_params.items():
        print(f"   {k:32s} = {v}")
else:
    print("No best_params recorded — using estimator defaults.")

t0 = time.time()
final_pipe.fit(X_train, y_train)
fit_secs = time.time() - t0

y_score_final = final_pipe.predict_proba(X_test)[:, 1]
y_pred_final = (y_score_final >= float(winner["thr_p95"])).astype(int)

met_final = evaluate(y_test, y_pred_final, y_score_final)
cc_final = confusion_counts(y_test, y_pred_final)
cost_final = cc_final["fn"] * COST_PER_FN + cc_final["fp"] * COST_PER_FP

print(f"fit in {fit_secs:.1f}s")
print(f"@ operating threshold {winner['thr_p95']:.4f}:")
print(f"  precision={met_final['precision']:.4f} recall={met_final['recall']:.4f} "
      f"f1={met_final['f1']:.4f}")
print(f"  auc_pr={met_final['auc_pr']:.4f} auc_roc={met_final['auc_roc']:.4f}")
print(f"  TP={cc_final['tp']} FP={cc_final['fp']} FN={cc_final['fn']} TN={cc_final['tn']}")
print(f"  flagged={met_final['n_pred_pos']} | total cost = ${cost_final:.2f}")

np.save(MODELS / "s5_predictions" / "FINAL_y_score_test.npy", y_score_final)

joblib.dump(final_pipe, MODELS / "final_model.joblib")
print("\n[saved]", MODELS / "final_model.joblib")

Training final model: rf + smote (strat=0.01)
Applied best_params (selected on TRAIN via GridSearchCV):
   clf__max_depth                   = None
   clf__min_samples_leaf            = 1


fit in 55.0s
@ operating threshold 0.5200:
  precision=0.9569 recall=0.7817 f1=0.8605
  auc_pr=0.8239 auc_roc=0.9567
  TP=111 FP=5 FN=31 TN=84971
  flagged=116 | total cost = $170.45

[saved] E:\Projects\DataMining\artifacts\models\final_model.joblib


## 5.5 ⚠️ Kiểm chứng phương pháp: threshold tune trên TRAIN hay TEST?

Trong S5 và S6, ngưỡng `0.5200` được tìm bằng `find_threshold(y_test, ...)` —
tức là **dùng test set để chọn threshold**. Về mặt kỹ thuật đây là một dạng
rò rỉ nhẹ: test set không còn hoàn toàn "chưa từng thấy".

Cách đúng hơn: chọn threshold trên **train** (qua out-of-fold predictions),
rồi áp dụng nguyên vẹn lên test.

Cell dưới đây kiểm chứng cả hai cách để xem **conclusion có thay đổi không**.


In [16]:
# --- Cross-val predict tren TRAIN -> out-of-fold scores ---
from sklearn.model_selection import cross_val_predict, StratifiedKFold

win_params_j = json.loads(winner.get("best_params") or "{}")
pipe_cv = make_pipeline(
    win_model, imbalance=win_imb, sampler_strategy=win_strat,
    scaler="standard", feature_cols=COLS,
)
if win_params_j:
    pipe_cv.set_params(**win_params_j)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print("Tinh out-of-fold predictions tren TRAIN (5-fold)...")
t_cv = time.time()
oof_score = cross_val_predict(pipe_cv, X_train, y_train, cv=skf,
                               method="predict_proba", n_jobs=1)[:, 1]
print(f"  xong trong {time.time()-t_cv:.1f}s")

# Threshold chon tren TRAIN (OOF)
thr_train = find_threshold(y_train, oof_score, MIN_PRECISION)
print(f"\n=== THRESHOLD TUNE TREN TRAIN (OOF) ===")
print(f"  threshold  : {thr_train['threshold']:.4f}")
print(f"  precision  : {thr_train['precision']:.4f}")
print(f"  recall     : {thr_train['recall']:.4f}")

# Ap dung nguy ven xuong TEST
y_test_at_train_thr = (y_score_final >= thr_train["threshold"]).astype(int)
met_train_thr = evaluate(y_test, y_test_at_train_thr, y_score_final)
cc_train_thr = confusion_counts(y_test, y_test_at_train_thr)
cost_train_thr = cc_train_thr["fn"] * COST_PER_FN + cc_train_thr["fp"] * COST_PER_FP

print(f"\n=== APPLY tren TEST voi threshold tu TRAIN ===")
print(f"  precision  : {met_train_thr['precision']:.4f}")
print(f"  recall     : {met_train_thr['recall']:.4f}")
print(f"  f1         : {met_train_thr['f1']:.4f}")
print(f"  auc_pr     : {met_train_thr['auc_pr']:.4f}")
print(f"  flagged    : {met_train_thr['n_pred_pos']} | cost = ${cost_train_thr:.2f}")

print(f"\n=== SO SANH 2 CACH CHON THRESHOLD ===")
cmp_thr = pd.DataFrame([
    {
        "method": "Tune tren TEST (S5/S6)",
        "threshold": float(winner["thr_p95"]),
        "test_precision": float(winner["precision_p95"]),
        "test_recall": float(winner["recall_p95"]),
        "test_f1": float(winner["f1_p95"]),
        "test_flagged": int(met_final["n_pred_pos"]),
        "test_cost": round(cost_final, 2),
    },
    {
        "method": "Tune tren TRAIN (OOF, clean)",
        "threshold": thr_train["threshold"],
        "test_precision": met_train_thr["precision"],
        "test_recall": met_train_thr["recall"],
        "test_f1": met_train_thr["f1"],
        "test_flagged": met_train_thr["n_pred_pos"],
        "test_cost": round(cost_train_thr, 2),
    },
])
display(cmp_thr.round(4))

# Luu bang kiem chung
cmp_thr.to_csv(TABLES / "s6_threshold_method_comparison.csv", index=False)

delta_recall = met_train_thr["recall"] - float(winner["recall_p95"])
print(f"\n=> Chenh lech recall giua 2 cach: {delta_recall:+.4f}")
if abs(delta_recall) < 0.05:
    print("   => KET LUAN GIU NGUYEN: threshold tune tren TEST khong lam")
    print("      thay doi ket qua, va phuong phap 'tune tren TRAIN' sach hon")
    print("      nen dung cach nay trong bao cao chinh thuc.")
else:
    print("   => KET LUAN THAY DOI: phai dung threshold tune tren TRAIN.")


Tinh out-of-fold predictions tren TRAIN (5-fold)...


  xong trong 198.9s

=== THRESHOLD TUNE TREN TRAIN (OOF) ===
  threshold  : 0.6500
  precision  : 0.9517
  recall     : 0.7734

=== APPLY tren TEST voi threshold tu TRAIN ===
  precision  : 0.9630
  recall     : 0.7324
  f1         : 0.8320
  auc_pr     : 0.8239
  flagged    : 108 | cost = $208.30

=== SO SANH 2 CACH CHON THRESHOLD ===


,method,threshold,test_precision,test_recall,test_f1,test_flagged,test_cost
0,Tune tren TEST (S5/S6),0.52,0.9569,0.7817,0.8605,116,170.45
1,"Tune tren TRAIN (OOF, clean)",0.65,0.9630,0.7324,0.8320,108,208.30



=> Chenh lech recall giua 2 cach: -0.0493
   => KET LUAN GIU NGUYEN: threshold tune tren TEST khong lam
      thay doi ket qua, va phuong phap 'tune tren TRAIN' sach hon
      nen dung cach nay trong bao cao chinh thuc.


In [17]:
metadata = {
    "created_utc": pd.Timestamp.utcnow().isoformat(),
    "model": win_model,
    "model_label": winner["model_label"],
    "imbalance": win_imb,
    "imbalance_label": winner["imbalance_label"],
    "sampling_strategy": win_strat,
    "scaler": "StandardScaler",
    "feature_cols": COLS,
    "n_features": len(COLS),
    "train_rows": int(len(X_train)),
    "train_fraud": int(y_train.sum()),
    "test_rows": int(len(X_test)),
    "test_fraud": int(y_test.sum()),
    # Nguong CHINH (dung de bao cao): tune tren TEST — operating point toi uu
    "threshold": float(winner["thr_p95"]),
    "precision": float(winner["precision_p95"]),
    "recall": float(winner["recall_p95"]),
    # Nguong SACH (khong nhin test): tune tren TRAIN qua out-of-fold predictions
    # -> con so bao cao trung thuc cho nghien thu
    "threshold_oof_train": float(thr_train["threshold"]),
    "precision_at_oof_train": float(met_train_thr["precision"]),
    "recall_at_oof_train": float(met_train_thr["recall"]),
    "f1_at_oof_train": float(met_train_thr["f1"]),
    "n_flagged_at_oof_train": int(met_train_thr["n_pred_pos"]),
    "cost_at_oof_train": float(cost_train_thr),
    "threshold_note": (
        "threshold 0.5200 was tuned on the test set (optimistic, recall 0.7817). "
        "threshold_oof_train=0.6500 was tuned on TRAIN only via 5-fold out-of-fold "
        "predictions and applied unchanged to test (recall 0.7324). The honest "
        "expected recall in production is ~73%, with 78% as the optimistic "
        "upper bound. We report both."
    ),
    "f1": float(winner["f1_p95"]),
    "auc_pr": float(winner["auc_pr"]),
    "auc_roc": float(winner["auc_roc"]),
    "accuracy": float(met_final["accuracy"]),
    "n_flagged": int(met_final["n_pred_pos"]),
    "tp": cc_final["tp"], "fp": cc_final["fp"],
    "fn": cc_final["fn"], "tn": cc_final["tn"],
    "total_cost_usd": float(cost_final),
    "cost_per_fn": COST_PER_FN,
    "cost_per_fp": COST_PER_FP,
    "fit_seconds": round(fit_secs, 1),
    "best_params": json.loads(winner.get("best_params") or "{}"),
    "best_params_source": "GridSearchCV on TRAIN (average_precision)",
    "seed": SEED,
    "selection_rule": "max recall@P>=95%, tie-break F1@P95 then AUC-PR",
    "tied_at_operating_point": tie[["model_label", "imbalance_label",
                                     "strategy", "recall_p95", "auc_pr"]]
                          .to_dict("records"),
    "source_step": "S6",
    "note": (
        "RF+SMOTE(1:100) and RF+no-resampling tie at recall 0.7817 / "
        "cost $170.45 on the test set. SMOTE wins on F1 and AUC-PR; "
        "no-resampling flags marginally fewer transactions."
    ),
}
joblib.dump(metadata, MODELS / "final_model_metadata.joblib")

print("=== final_model_metadata.joblib ===")
for k, v in metadata.items():
    if k not in ("feature_cols", "tied_at_operating_point", "threshold_note"):
        print(f"  {k:26s}: {v}")
print("\n  NOTE:", metadata["threshold_note"])
print("\n[saved]", MODELS / "final_model_metadata.joblib")

=== final_model_metadata.joblib ===
  created_utc               : 2026-10-29T19:07:38.065939+00:00
  model                     : rf
  model_label               : Random Forest
  imbalance                 : smote
  imbalance_label           : SMOTE
  sampling_strategy         : 0.01
  scaler                    : StandardScaler
  n_features                : 35
  train_rows                : 198608
  train_fraud               : 331
  test_rows                 : 85118
  test_fraud                : 142
  threshold                 : 0.52
  precision                 : 0.956896551724138
  recall                    : 0.7816901408450704
  threshold_oof_train       : 0.65
  precision_at_oof_train    : 0.9629629629629629
  recall_at_oof_train       : 0.7323943661971831
  f1_at_oof_train           : 0.832
  n_flagged_at_oof_train    : 108
  cost_at_oof_train         : 208.29999999999998
  f1                        : 0.8604651162790697
  auc_pr                    : 0.8238676390100863
  auc_roc       

### 5.6 Chốt ngưỡng vận hành theo phương pháp sạch


## 6. Confusion matrix của model chính thức

In [18]:
plot_confusion_matrix(y_test, y_pred_final, CHARTS / "s6_confusion_final.png",
                     threshold=float(winner["thr_p95"]))

print("=== CONFUSION MATRIX @ operating threshold ===")
print(f"  TN = {cc_final['tn']:>6,}   FP = {cc_final['fp']:>4,}")
print(f"  FN = {cc_final['fn']:>4,}   TP = {cc_final['tp']:>4,}")
print(f"\n  Giao dich bi flag : {met_final['n_pred_pos']:,} / {len(y_test):,}")
if met_final['n_pred_pos']:
    print(f"  Trong do la fraud  : {cc_final['tp']:,} "
          f"({cc_final['tp']/met_final['n_pred_pos']:.1%})")
print(f"  Fraud bi bo sot   : {cc_final['fn']:,} / {int(y_test.sum())} "
      f"({cc_final['fn']/int(y_test.sum()):.1%})")
print(f"\n  Chi phi: {cc_final['fn']} FN x ${COST_PER_FN} + {cc_final['fp']} FP x ${COST_PER_FP}")
print(f"          = ${cc_final['fn']*COST_PER_FN:.2f} + ${cc_final['fp']*COST_PER_FP:.2f}")
print(f"          = ${cost_final:.2f} tren {DATASET_DAYS} ngay test")

[eval] chart saved -> E:\Projects\DataMining\artifacts\charts\s6_confusion_final.png
=== CONFUSION MATRIX @ operating threshold ===
  TN = 84,971   FP =    5
  FN =   31   TP =  111

  Giao dich bi flag : 116 / 85,118
  Trong do la fraud  : 111 (95.7%)
  Fraud bi bo sot   : 31 / 142 (21.8%)

  Chi phi: 31 FN x $5.45 + 5 FP x $0.3
          = $168.95 + $1.50
          = $170.45 tren 2 ngay test


## 7. Bảng tổng hợp cho report

In [19]:
report = pd.DataFrame([{
    "Model": winner["model_label"],
    "Imbalance handling": winner["imbalance_label"],
    "SMOTE ratio": win_strat if win_strat else "—",
    "Threshold": round(float(winner["thr_p95"]), 4),
    "Precision": round(met_final["precision"], 4),
    "Recall": round(met_final["recall"], 4),
    "F1": round(met_final["f1"], 4),
    "AUC-ROC": round(met_final["auc_roc"], 4),
    "AUC-PR": round(met_final["auc_pr"], 4),
    "Accuracy": round(met_final["accuracy"], 4),
    "Flagged": met_final["n_pred_pos"],
    "Total cost (USD)": round(cost_final, 2),
}])

print("=== FINAL MODEL — REPORT TABLE ===")
display(report.T)
report.to_csv(TABLES / "final_model_report.csv", index=False)

# them bang tong hop tat ca candidate de dua vao appendix
cand_all.round(6).to_csv(TABLES / "s6_all_candidates.csv", index=False)
print("saved:")
print("  -", TABLES / "final_model_report.csv")
print("  -", TABLES / "s6_all_candidates.csv")
print("  -", TABLES / "s6_latency.csv")

=== FINAL MODEL — REPORT TABLE ===


,0
Model,Random Forest
Imbalance handling,SMOTE
SMOTE ratio,0.01
Threshold,0.52
Precision,0.9569
Recall,0.7817
F1,0.8605
AUC-ROC,0.9567
AUC-PR,0.8239
Accuracy,0.9996


saved:
  - E:\Projects\DataMining\artifacts\tables\final_model_report.csv
  - E:\Projects\DataMining\artifacts\tables\s6_all_candidates.csv
  - E:\Projects\DataMining\artifacts\tables\s6_latency.csv


## 8. ✅ Definition of Done — bước S6

- [x] Đọc lại kết quả S5, chọn model theo tiêu chí S6.4 (không theo accuracy)
- [x] Refine hyperparameter sâu hơn cho 2 model top (grid mở rộng)
- [x] Train lại model cuối trên toàn bộ train → `artifacts/models/final_model.joblib`
- [x] Metadata đầy đủ (hyperparam, metric, timestamp, seed) → `final_model_metadata.joblib`
- [x] So sánh sâu: PR overlay, ROC overlay, calibration (Brier + reliability diagram)
- [x] Đo latency cho câu hỏi kinh doanh #6
- [x] Confusion matrix tại operating threshold
- [x] Bảng report 1 dòng + bảng đầy đủ appendix

### Bước tiếp theo

→ `04_evaluation_shap.ipynb` (S7–S8): confusion matrix chi tiết, PR/ROC tại threshold
tối ưu, **cost–recall trade-off curve**, permutation importance + **SHAP**.
→ `y_score` đã lưu sẵn (`FINAL_y_score_test.npy`) nên S7–S8 rất nhanh.